# BioEvidence Research Agent

**Author:** Aruna Kumar

A **LangGraph** research agent for biomedical questions. It queries **real APIs** (OpenTargets, PubMed, PubMed Central, UniProt) through ToolUniverse, uses **DuckDuckGo** for web context, and ranks literature passages with **BM25** before writing a structured markdown report.

---

## Architecture

```
User question
      |
      v
 +----------+   tool_calls    +----------------+
 | PLANNER  |---------------->| TOOL EXECUTOR  |
 | (LLM+7)  |<----------------| (runs tools)   |
 +-----+----+   ToolMessages  +----------------+
       |
       | no more tool_calls
       v
 +-------------+
 | SYNTHESIZER |---> Final markdown report
 | (LLM only)  |
 +-------------+

Tools: disease_lookup, disease_evidence, fetch_articles,
       build_chunks, bm25_search, protein_info, web_search
```

---

## Tech stack

| Layer | Technology | Role |
|-------|------------|------|
| **Orchestration** | LangGraph `StateGraph` | Planner / executor / synthesizer loop |
| **LLM** | Azure OpenAI | Planning, tool calls, report writing |
| **APIs** | ToolUniverse | OpenTargets, PubMed, UniProt wrappers |
| **Web** | `ddgs` (DuckDuckGo) | Supplementary snippets (no API key) |
| **Literature** | `rank_bm25` + files on disk | Keyword search over article text |

> Credentials and deployment names live in **`.env`** — see `.env.example`.



In [ ]:
# Run once if needed
%pip install rank-bm25 ddgs -q

## 1 — Environment setup

| Piece | Role |
|-------|------|
| **`AzureChatOpenAI`** | Planner (with tools) and synthesizer (plain LLM) |
| **`ToolUniverse`** | Loads biomedical tool definitions for OpenTargets, PubMed, UniProt |
| **`literature_bm25`** | Helper file with four functions for the BM25 literature pipeline |
| **`lit_workspace/`** | Folder where `catalog.json`, `chunks.jsonl`, and `bm25_evidence.txt` are saved |

> Tip: keep API keys and Azure deployment IDs only in `.env`.



In [ ]:
import os
import re
import time
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import AzureChatOpenAI
from langgraph.graph import END, START, MessagesState, StateGraph
from IPython.display import Image, display, Markdown
from tooluniverse import ToolUniverse
from ddgs import DDGS

# ── Import our BM25 helper functions from literature_bm25.py ───────────────
# These are plain Python functions — the @tool wrappers are defined below.
# This is how you connect a helper file to a LangGraph agent:
#   1. Import the functions
#   2. Wrap each one with @tool
#   3. Add to the tools list
from literature_bm25 import (
    literature_collect,
    literature_build_chunks,
    literature_bm25_search,
    reset_literature_workspace,
)

# ── Load environment variables from .env ───────────────────────────────────
load_dotenv()

# ── LLM — the brain of our agent ──────────────────────────────────────────
llm = AzureChatOpenAI(
    azure_deployment=os.getenv("AZURE_OPENAI_LLM_MODEL_DEPLOYMENT_ID"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

# ── ToolUniverse — loads 2,000+ biomedical API tools ──────────────────────
tu = ToolUniverse()
tu.load_tools()

# ── Workspace for BM25 intermediate files ─────────────────────────────────
# Each research() run saves 3 files here:
#   catalog.json       ← article metadata from PubMed
#   chunks.jsonl       ← chunked article text
#   bm25_evidence.txt  ← top-k ranked passages (what the LLM uses)
lit_workspace = Path("lit_workspace")
lit_workspace.mkdir(exist_ok=True)

print(f" LLM ready")
print(f" ToolUniverse ready — {len(tu.tools)} tools loaded")
print(f" BM25 workspace: {lit_workspace.resolve()}")

In [ ]:
# Quick Sanity Check — make sure LLM and ToolUniverse are working

# 1. Can the LLM respond?
test_llm = llm.invoke("Hi, How are you.")
print(f"LLM test: {test_llm.content}")

# 2. Can ToolUniverse call a real API?
test_tu = tu.run_one_function({
    "name": "OpenTargets_get_disease_id_description_by_name",
    "arguments": {"diseaseName": "Alzheimer's disease"},
})
hits = test_tu.get("data", {}).get("search", {}).get("hits", [])
print(f"ToolUniverse test: found {len(hits)} results ✓")

print("\n All systems working — let's build the tools!")

## 2 — Tools (7 functions)

Each tool is wrapped with LangChain's `@tool` decorator so the LLM receives its name, description, and input schema.

We define and smoke-test each tool before composing the graph so failures are easy to localize.

| # | Tool | Source | Purpose |
|---|------|--------|----------|
| 1 | `disease_lookup` | OpenTargets | Disease name -> EFO/MONDO ID |
| 2 | `disease_evidence` | OpenTargets | Disease ID -> associated genes + scores |
| 3 | `fetch_articles` | PubMed | Search -> save catalog.json (IDs, titles, abstracts, PMC IDs) |
| 4 | `build_chunks` | PubMed Central + local | Download free full text or use abstract -> chunk -> chunks.jsonl |
| 5 | `bm25_search` | rank_bm25 | Keyword-rank chunks -> save bm25_evidence.txt |
| 6 | `protein_info` | UniProt | Accession -> protein function summary |
| 7 | `web_search` | DuckDuckGo | Web snippets for trials, news, context |

**Tools 3-5 form the BM25 literature pipeline** and must be called in order:
`fetch_articles` -> `build_chunks` -> `bm25_search`

---



In [ ]:
# ══════════════════════════════════════════════════════════════════════
# TOOL 1: disease_lookup
# Given a disease name → find its official EFO/MONDO ID from OpenTargets
# The agent always calls this first so it has the right ID for tool 2
# ══════════════════════════════════════════════════════════════════════

@tool
def disease_lookup(disease_name: str) -> str:
    """Look up a disease by name to get its EFO/MONDO ID and description
    from the OpenTargets database.

    Args:
        disease_name: Name of the disease (e.g. "Alzheimer's disease", "breast cancer")
    """
    result = tu.run_one_function({
        "name": "OpenTargets_get_disease_id_description_by_name",
        "arguments": {"diseaseName": disease_name},
    })
    return str(result)

print(f" Defined: {disease_lookup.name}")

In [ ]:
# Test it
result = disease_lookup.invoke({"disease_name": "Parkinson's disease"})
print("disease_lookup result (first 400 chars):")
print(result[:400])

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# TOOL 2: disease_evidence
# Given a disease ID → fetch the top associated gene targets
# We extract just the top 10 to keep responses manageable for the LLM
# ══════════════════════════════════════════════════════════════════════

@tool
def disease_evidence(efo_id: str) -> str:
    """Fetch gene targets associated with a disease from OpenTargets.
    Returns the top 10 targets with their association scores.

    Args:
        efo_id: Disease EFO/MONDO ID (e.g. "MONDO_0004975" for Alzheimer's)
    """
    result = tu.run_one_function({
        "name": "OpenTargets_get_associated_targets_by_disease_efoId",
        "arguments": {"efoId": efo_id},
    })
    if isinstance(result, dict) and "data" in result:
        targets = result["data"].get("disease", {}).get("associatedTargets", {})
        rows = targets.get("rows", [])[:10]
        summary = {
            "total_targets": targets.get("count", 0),
            "top_10": [
                {
                    "gene": r.get("target", {}).get("approvedSymbol", "?"),
                    "ensembl_id": r.get("target", {}).get("id", "?"),
                    "score": round(r.get("score", 0), 4),
                }
                for r in rows
            ],
        }
        return str(summary)
    return str(result)

print(f"✅ Defined: {disease_evidence.name}")

In [ ]:
# Test with Parkinson's disease MONDO ID
result = disease_evidence.invoke({"efo_id": "MONDO_0005180"})
print("disease_evidence result (first 400 chars):")
print(result[:400])

### Tools 3, 4, 5 — BM25 Literature Pipeline

These three tools handle literature retrieval using keyword-based (BM25) ranking over PubMed article text.

**Why BM25?** Gene names, drug names, and protein identifiers are exact strings. BM25 (a word-frequency scoring algorithm) matches them directly without needing an embeddings API, and the intermediate files are easy to inspect.

**How the three tools connect:**

```
fetch_articles(query)
    -> PubMed search -> saves lit_workspace/catalog.json
    -> returns "/path/to/catalog.json"
           |
build_chunks(catalog_path)
    -> for each article: try PMC free full text -> fall back to abstract
    -> chunk text -> saves lit_workspace/chunks.jsonl
    -> returns "/path/to/chunks.jsonl + summary"
           |
bm25_search(chunks_path, keywords)
    -> BM25 keyword ranking -> saves lit_workspace/bm25_evidence.txt
    -> returns "/path/to/bm25_evidence.txt + preview"
```

The file paths flow from one tool to the next. **All heavy text stays on disk** — only short paths pass through the LLM messages.



In [ ]:
# ══════════════════════════════════════════════════════════════════════
# TOOL 3: fetch_articles
# Search PubMed → save catalog.json with article metadata
# This is a thin @tool shell — real logic is in literature_bm25.py
# ══════════════════════════════════════════════════════════════════════

@tool
def fetch_articles(query: str, max_articles: int = 8) -> str:
    """Search PubMed for biomedical articles and save them to catalog.json.
    Always call this FIRST in the literature pipeline, before build_chunks.

    Args:
        query: Biomedical search terms (e.g. "LRRK2 kinase Parkinson's disease")
        max_articles: How many articles to fetch (default 8, max 25)

    Returns:
        Absolute path to catalog.json — pass this to build_chunks.
    """
    reset_literature_workspace(lit_workspace)
    return literature_collect(tu, query, max_articles, lit_workspace)

print(f" Defined: {fetch_articles.name}")

In [ ]:
# Test Tool 3
import json
catalog_path = fetch_articles.invoke({"query": "LRRK2 Parkinson disease gene", "max_articles": 3})
print(f"catalog_path: {catalog_path}")

catalog_data = json.loads(Path(catalog_path).read_text())
print(f"Articles found: {catalog_data['count']}")
first = catalog_data["articles"][0]
print(f"First article PMID: {first['pmid']}")
print(f"PMC ID (free full text?): {first['pmc_id'] or 'None (abstract only)'}")
print(f"Title: {first['title'][:80]}...")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# TOOL 4: build_chunks
# Read catalog.json → try PMC full text (FREE) → fall back to abstract
# → chunk text into overlapping 420-char pieces → save chunks.jsonl
#
# Why 420 chars with 80-char overlap?
#   - BM25 works best on focused short pieces
#   - Overlap means no sentence gets cut in half at a boundary
# ══════════════════════════════════════════════════════════════════════

@tool
def build_chunks(catalog_path: str) -> str:
    """Read catalog.json, fetch free PMC full text where available,
    fall back to abstract for paywalled articles, chunk everything.
    Call this AFTER fetch_articles, BEFORE bm25_search.

    Args:
        catalog_path: Path returned by fetch_articles

    Returns:
        Path to chunks.jsonl (first line) + summary of open-access coverage.
    """
    return literature_build_chunks(tu, catalog_path, lit_workspace)

print(f" Defined: {build_chunks.name}")

In [ ]:
# Test Tool 4
chunks_result = build_chunks.invoke({"catalog_path": catalog_path})
print(chunks_result)

# The path is always the first line
chunks_path = chunks_result.split("\n")[0].strip()
print(f"\nFirst chunk preview:")
with open(chunks_path) as f:
    first_chunk = json.loads(f.readline())
print(f"  Source: {first_chunk['text_source']}")
print(f"  Text: {first_chunk['text'][:150]}...")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# TOOL 5: bm25_search
# Load chunks → BM25 keyword scoring → save top-k passages to file
#
# IMPORTANT: search_query should be SHORT keywords (3-6 words)!
# BM25 is keyword-based — a long sentence dilutes the scores.
#   Good:  "LRRK2 kinase dopamine Parkinson"
#   Bad:   "What is the role of LRRK2 in Parkinson's disease?"
# ══════════════════════════════════════════════════════════════════════

@tool
def bm25_search(chunks_path: str, search_query: str, top_k: int = 6) -> str:
    """Run BM25 keyword search over chunks and save the top passages.
    Call this AFTER build_chunks. This is the final step of the literature pipeline.

    Args:
        chunks_path: Path returned by build_chunks (use the first line only)
        search_query: 3-6 key biomedical keywords from the question.
                      Use short terms NOT the full question!
        top_k: How many top passages to retrieve (default 6)

    Returns:
        Path to bm25_evidence.txt (first line) + ranked passage preview.
    """
    return literature_bm25_search(chunks_path, search_query, top_k)

print(f" Defined: {bm25_search.name}")

In [ ]:
# Test Tool 5
evidence_result = bm25_search.invoke({
    "chunks_path": chunks_path,
    "search_query": "LRRK2 kinase Parkinson dopamine neuron",
    "top_k": 3,
})
print(evidence_result[:600])

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# TOOL 6: protein_info
# Get detailed protein information from UniProt — name, gene, function
# The agent calls this when it needs specifics about a protein target
# ══════════════════════════════════════════════════════════════════════

@tool
def protein_info(uniprot_id: str) -> str:
    """Get protein details from UniProt — name, gene, organism, and function.

    Args:
        uniprot_id: UniProt accession ID (e.g. "P05067" for Amyloid-beta precursor)
    """
    result = tu.run_one_function({
        "name": "UniProt_get_entry_by_accession",
        "arguments": {"accession": uniprot_id},
    })
    if isinstance(result, dict):
        summary = {
            "protein": result.get("uniProtkbId", ""),
            "accession": result.get("primaryAccession", ""),
            "gene": result.get("genes", [{}])[0].get("geneName", {}).get("value", ""),
            "organism": result.get("organism", {}).get("scientificName", ""),
            "function": "",
        }
        for comment in result.get("comments", []):
            if comment.get("commentType") == "FUNCTION":
                texts = comment.get("texts", [])
                if texts:
                    summary["function"] = texts[0].get("value", "")[:500]
                break
        return str(summary)
    return str(result)

print(f" Defined: {protein_info.name}")

In [ ]:
# Test with P05067 (Amyloid-beta precursor protein — central to Alzheimer's)
result = protein_info.invoke({"uniprot_id": "P05067"})
print("protein_info result:")
print(result[:400])

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# TOOL 7: web_search
# General web search via DuckDuckGo — free, no API key needed
# The agent uses this for clinical trials, news, and recent context
# ══════════════════════════════════════════════════════════════════════

@tool
def web_search(query: str) -> str:
    """Search the web for additional context — clinical trials, news, treatments.
    Good for information not in biomedical databases.

    Args:
        query: Search query (e.g. "latest LRRK2 inhibitor clinical trials 2026")
    """
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))
        return str([
            {"title": r.get("title", ""), "body": r.get("body", ""), "url": r.get("href", "")}
            for r in results
        ])
    except Exception as e:
        return f"Web search failed: {e}."

print(f" Defined: {web_search.name}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Register all 7 tools with the LLM
#
# bind_tools() injects tool schemas into every LLM request — this is
# how the model knows what tools exist and how to call them.
#
# tools_by_name is a lookup dict so tool_executor can find the right
# function when the LLM asks for it.
# ══════════════════════════════════════════════════════════════════════

tools = [
    disease_lookup,    # OpenTargets: disease name → MONDO/EFO ID
    disease_evidence,  # OpenTargets: ID → top gene targets + scores
    fetch_articles,    # PubMed Step 1: search → catalog.json
    build_chunks,      # PubMed Step 2: PMC full text or abstract → chunks.jsonl
    bm25_search,       # PubMed Step 3: BM25 keyword rank → bm25_evidence.txt
    protein_info,      # UniProt: accession → protein function
    web_search,        # DuckDuckGo: news, trials, context
]

tools_by_name = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

print(f" Registered {len(tools)} tools with the LLM:")
for t in tools:
    print(f"   • {t.name}")

## 3 — Agent nodes (planner, executor, synthesizer)

Three nodes and one router function:

| Node | Does what |
|------|-----------|
| **Planner** | Reads all messages + `PLANNER_PROMPT`; decides which tool to call next |
| **Tool executor** | Runs each requested tool; errors become string messages (no crash) |
| **Synthesizer** | Plain LLM (no tools) with `SYNTHESIZER_PROMPT` — writes the final report |
| **`should_continue`** | Has `tool_calls`? -> executor. No `tool_calls`? -> synthesizer |

The `PLANNER_PROMPT` instructs the model to use the three literature tools in order (`fetch_articles` -> `build_chunks` -> `bm25_search`) and to call `disease_lookup` first.



In [ ]:
# ══════════════════════════════════════════════════════════════════════
# System Prompts — the "job descriptions" for the two LLM nodes
# ══════════════════════════════════════════════════════════════════════

PLANNER_PROMPT = """\
You are a biomedical research assistant with access to 7 tools.
For any research question, gather evidence using this strategy:

1. IDENTIFY — use disease_lookup to get the official disease ID
2. EVIDENCE — use disease_evidence with that ID to find gene targets
3. LITERATURE — run these 3 tools IN ORDER (each uses the previous output):
   a. fetch_articles(query, max_articles=8)
      Use main disease/gene terms as the query.
   b. build_chunks(catalog_path)
      Pass the EXACT path returned by fetch_articles.
   c. bm25_search(chunks_path, search_query, top_k=6)
      - chunks_path: FIRST LINE of output from build_chunks
      - search_query: extract 3-6 KEY TERMS from the question (not the full question!)
        Example: question="What is LRRK2's role in Parkinson's?"
                 search_query="LRRK2 kinase Parkinson dopamine"
4. DETAILS — use protein_info for 1-2 top genes from step 2
5. WEB — use web_search for clinical trials, news, recent context

Rules:
- Always run disease_lookup FIRST
- Always run the 3 literature tools in order: fetch_articles → build_chunks → bm25_search
- Stop when you have gathered from at least 4 sources
- Keep tool arguments concise
"""

SYNTHESIZER_PROMPT = """\
You are a biomedical research report writer.
Based on ALL the research data gathered in this conversation, write a clear report.

Format your report exactly like this:

## Disease Overview
Brief description of the disease.

## Top Gene Targets
| Gene Symbol | Ensembl ID | Association Score |
|-------------|-----------|-------------------|
(Fill from disease_evidence data. Include up to 10 genes.)

## Literature Findings
Summarize the most relevant passages from the BM25 search.
Cite PMIDs like (PMID: 12345678). Note if source was full text or abstract only.

## Additional Insights
Findings from web search, protein info, or cross-referencing.

## Summary
A concise 2-3 sentence conclusion answering the user's original question.

Rules:
- ONLY include information that was actually retrieved — never make up data
- If a section has no data, say "No data retrieved for this section"
"""

print("✅ System prompts defined")
print(f"   Planner prompt:     {len(PLANNER_PROMPT)} chars")
print(f"   Synthesizer prompt: {len(SYNTHESIZER_PROMPT)} chars")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# NODE 1: planner
# Sends the full conversation to the LLM (with tool schemas attached)
# The LLM either requests tool calls or stops (meaning it has enough data)
# ══════════════════════════════════════════════════════════════════════

def planner(state: MessagesState):
    system = SystemMessage(content=PLANNER_PROMPT)
    response = llm_with_tools.invoke([system] + state["messages"])
    return {"messages": [response]}


# ══════════════════════════════════════════════════════════════════════
# NODE 2: tool_executor
# Runs every tool the planner asked for and returns ToolMessage results
# Wrapped in try/except so one failed tool doesn't crash the whole agent
# ══════════════════════════════════════════════════════════════════════

def tool_executor(state: MessagesState):
    results = []
    for call in state["messages"][-1].tool_calls:
        try:
            tool_fn = tools_by_name[call["name"]]
            output = tool_fn.invoke(call["args"])
        except Exception as e:
            output = f"Error calling {call['name']}: {e}"
        results.append(
            ToolMessage(content=str(output), tool_call_id=call["id"])
        )
    return {"messages": results}


# ══════════════════════════════════════════════════════════════════════
# NODE 3: synthesizer
# Uses plain llm (NO tools) — just reads all messages and writes a report
# Separating gathering (planner+tools) from writing (synthesizer) gives
# cleaner, more focused reports
# ══════════════════════════════════════════════════════════════════════

def synthesizer(state: MessagesState):
    system = SystemMessage(content=SYNTHESIZER_PROMPT)
    response = llm.invoke([system] + state["messages"])   # plain llm, no tools!
    return {"messages": [response]}


# ══════════════════════════════════════════════════════════════════════
# ROUTER: should_continue
# Has tool_calls? → tool_executor (keep researching)
# No tool_calls?  → synthesizer (write the report)
# ══════════════════════════════════════════════════════════════════════

def should_continue(state: MessagesState):
    if state["messages"][-1].tool_calls:
        return "tool_executor"
    return "synthesizer"


print("✅ Defined: planner, tool_executor, synthesizer, should_continue")

## 4 — Wire the `StateGraph`

```
START -> planner -> should_continue -> tool_executor -> planner  (loop)
                                    -> synthesizer -> END
```

- `MessagesState` carries the growing chat history (human, assistant, tool).
- `tool_executor` always returns to `planner` so the model sees results and decides the next step.
- `synthesizer` is terminal — no further tool calls.



In [ ]:
# Build the state graph
graph = StateGraph(MessagesState)

# Add nodes
graph.add_node("planner",       planner)
graph.add_node("tool_executor", tool_executor)
graph.add_node("synthesizer",   synthesizer)

# Wire edges
graph.add_edge(START, "planner")
graph.add_conditional_edges("planner", should_continue, {
    "tool_executor": "tool_executor",
    "synthesizer":   "synthesizer",
})
graph.add_edge("tool_executor", "planner")   # always loop back to planner
graph.add_edge("synthesizer",   END)

agent = graph.compile()
print("✅ Agent compiled!")

try:
    display(Image(agent.get_graph(xray=True).draw_mermaid_png()))
except Exception:
    print(agent.get_graph().draw_mermaid())

## 5 — Run a full question

The `research()` helper:

1. Resets `lit_workspace/` so each question starts with clean files.
2. Runs the agent with `agent.invoke()` and a generous `recursion_limit`.
3. Prints a trace of every tool call.
4. Lists the BM25 files created in `lit_workspace/`.
5. Displays the final report as rendered markdown.
6. Optionally saves a timestamped markdown file to `reports/`.

> **Tip:** After a run, open `lit_workspace/bm25_evidence.txt` to see exactly what the LLM used for the literature section.



In [ ]:
REPORTS_DIR = "reports"
Path(REPORTS_DIR).mkdir(exist_ok=True)


def _make_report_name(question: str) -> str:
    noise = ["what", "is", "the", "of", "and", "in", "are", "a", "an",
             "does", "do", "find", "recent", "summarize", "about", "how"]
    words = re.sub(r"[^a-zA-Z0-9\s]", "", question).split()
    keywords = [w.capitalize() for w in words if w.lower() not in noise][:5]
    return "_".join(keywords) if keywords else "Research"


def research(question: str, save_report: bool = True):
    """Run the agent, print a trace, show the report, optionally save markdown."""
    reset_literature_workspace(lit_workspace)
    print(f"🔬 Research question:\n   {question}")
    print("=" * 72)

    t0 = time.time()
    response = agent.invoke(
        {"messages": [HumanMessage(question)]},
        config={"recursion_limit": 30},
    )
    elapsed = time.time() - t0

    # ── Print trace ───────────────────────────────────────────────────
    trace_lines = []
    step = 1
    for msg in response["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            called = [tc["name"] for tc in msg.tool_calls]
            trace_lines.append(f"  Step {step}: planner → {called}")
            step += 1
        elif isinstance(msg, ToolMessage):
            preview = msg.content[:120] + "..." if len(msg.content) > 120 else msg.content
            trace_lines.append(f"           ↳ {preview}")

    print(f"\n⏱️  Completed in {elapsed:.1f}s ({step-1} planner tool round(s))")
    print("\n📋 Trace")
    print("-" * 40)
    for line in trace_lines:
        print(line)

    # ── Show BM25 files created ───────────────────────────────────────
    print("\n📁 BM25 files in lit_workspace/:")
    for f in sorted(lit_workspace.iterdir()):
        print(f"   {f.name:25} ({f.stat().st_size:,} bytes)")

    # ── Display report ────────────────────────────────────────────────
    final_report = response["messages"][-1].content
    print("\n" + "=" * 72 + "\n📄 FINAL REPORT\n" + "=" * 72 + "\n")
    display(Markdown(final_report))

    if save_report:
        date_str = datetime.now().strftime("%Y-%m-%d")
        filename = f"{REPORTS_DIR}/{_make_report_name(question)}_{date_str}.md"
        Path(filename).write_text(
            f"# Research report\n\n**Query:** {question}\n\n"
            f"**Date:** {datetime.now().strftime('%B %d, %Y at %H:%M')}\n\n"
            f"**Runtime:** {elapsed:.1f}s\n\n---\n\n"
            + "\n".join(trace_lines)
            + "\n\n---\n\n" + str(final_report),
            encoding="utf-8",
        )
        print(f"\n💾 Report saved → {filename}")

    return response


print("✅ research() helper defined — ready to run!")

In [ ]:
# ── Example run: Parkinson's disease ──────────────────────────────────────
# Watch the full pipeline:
#   disease_lookup → disease_evidence → fetch_articles → build_chunks
#   → bm25_search → protein_info → web_search → synthesizer

response = research(
    "What gene targets are associated with Parkinson's disease "
    "and what does recent research say about them?"
)

In [ ]:
# ── Example run 2: BRCA1 & Breast Cancer ──────────────────────────────────
response2 = research(
    "What is the role of BRCA1 and BRCA2 genes in breast cancer? "
    "Find recent publications and summarize the evidence."
)

---

## Summary

### End-to-end flow

```
Question -> planner <-> tool_executor (7 tools) -> synthesizer -> markdown report
```

### Design choices

| Idea | Why it matters |
|------|----------------|
| **Small tools** | Each tool does one thing — easier to test, debug, and swap |
| **BM25 on disk** | Intermediate files (`catalog.json`, `chunks.jsonl`, `bm25_evidence.txt`) are human-readable and inspectable |
| **Separate synthesizer** | The planner gathers data; the synthesizer writes the report — cleaner output |
| **Helper file pattern** | `literature_bm25.py` has no LangGraph dependency, so each function can be tested standalone |
| **Fault isolation** | `try/except` in the executor keeps one bad API response from killing the run |

### To explore further

- Open `lit_workspace/bm25_evidence.txt` to see what BM25 found
- Open `lit_workspace/catalog.json` to see all fetched articles
- Try increasing `max_articles` to 15 for more coverage
- Run `streamlit run app.py` for the interactive web interface

